# UC4 BRDF sweep playground

This notebook demonstrates the angle-resolved scattering workflow for a rough surface.

What this notebook is showing:
1. A rough surface is illuminated from a defined incidence angle.
2. The scattering model is evaluated across a range of reflection angles.
3. The sweep is summarised and fitted with a simple BRDF-like model.
4. The result is compared across different geometric settings so you can see how the response changes.


The goal is not to reproduce a full scatterometer exactly, but to make the core angle-resolved scattering workflow feel tangible and editable.Change the parameters in the next cell and rerun the later cells to see how the sweep changes.


## How to use this notebook

- Edit the parameters in the next cell to change the sweep range, scattering model, or surface roughness.
- Run the cells in order so you can see how each stage changes the result.
- If the fit looks poor, that is useful: it usually means the chosen geometry is pushing the simple model outside its comfort zone.
- Use the printed summary metrics as a guide to understand what changed.

In [ ]:
# Editable parameters: change these values and rerun the notebook.
theta_i_range = (0.0, 0.8, 4)
theta_r_range = (0.0, 0.8, 4)
phi_range = (0.0, 0.0, 1)

print("Configuration:")
print(f"  theta_i_range={theta_i_range}")
print(f"  theta_r_range={theta_r_range}")
print(f"  phi_range={phi_range}")

In [ ]:
import numpy as np

from optical_metrology.analysis import BRDFFitter, GoniometricSweep
from optical_metrology.illumination import Laser
from optical_metrology.scattering import BeckmannScattering
from optical_metrology.surface import Material, RoughSurface

In [ ]:
# Create a rough surface and a light source, then run the sweep.
surface = RoughSurface(shape=(16, 16), sigma=2.0, amplitude=0.2, material=Material("silicon"))
source = Laser(wavelength=550e-9, power=1.0)
source.propagation_direction = np.array([0.0, 0.0, -1.0])
source.direction = np.array([0.0, 0.0, -1.0])
model = BeckmannScattering(roughness=0.2)
sweep = GoniometricSweep(theta_i_range=theta_i_range, theta_r_range=theta_r_range, phi_range=phi_range)
measurements = sweep.sweep(model, source, surface, np.array([0.0, 0.0, 1.0]))

print(f"Collected {len(measurements)} measurements")

In [ ]:
# Summarise the sweep and fit a simple BRDF-like model.
report = sweep.analyze([(m.theta_i, m.theta_r, m.phi_i, m.phi_r, m.brdf) for m in measurements])


def model_fn(theta_i, theta_r, phi_i, phi_r, roughness, scale):
    return np.full_like(theta_i, scale, dtype=float) * np.exp(-0.5 * np.square(theta_r / max(roughness, 1e-6)))

fitter = BRDFFitter(
    model_fn=model_fn,
    initial_params={"roughness": 0.2, "scale": 1.0},
    param_bounds={"roughness": (1e-3, 1.0), "scale": (1e-6, 10.0)},
)
fit_report = fitter.analyze(report.measurements["measurements"])

print("Sweep summary:")
for key, value in report.measurements.items():
    if key != "measurements":
        print(f"  {key}: {value}")

print("\nFit summary:")
for key, value in fit_report.measurements.items():
    print(f"  {key}: {value}")

## Try next

Try changing one thing at a time:
- widen the incidence or reflection angle range to see how the sweep broadens
- change the roughness parameter in the scattering model to see how the response changes
- compare the results for a smoother or rougher surface

- keep the surface fixed and vary only the incidence angle so the effect is easier to isolateA good experiment is to hold the surface constant and vary only the incidence angle; the pattern usually changes in a very interpretable way.
